In [ ]:
import pandas as pd
import sqlite3
import zipfile

zip_path = "/content/archive (1).zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/olist")

print("Files extracted:")
print(zip_ref.namelist())

Files extracted:
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [ ]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'olist', 'archive (1).zip']


In [ ]:
import pandas as pd

path = "/content/olist/"

orders = pd.read_csv(path + "olist_orders_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
customers = pd.read_csv(path + "olist_customers_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")

In [ ]:
print("Orders:", len(orders))
print("Order Items:", len(order_items))
print("Customers:", len(customers))
print("Products:", len(products))
print("Sellers:", len(sellers))
print("Payments:", len(payments))
print("Reviews:", len(reviews))

Orders: 99441
Order Items: 112650
Customers: 99441
Products: 32951
Sellers: 3095
Payments: 103886
Reviews: 99224


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")

print("SQL connection established.")

SQL connection established.


In [ ]:
orders.to_sql("orders", conn, index=False, if_exists="replace")
order_items.to_sql("order_items", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")
sellers.to_sql("sellers", conn, index=False, if_exists="replace")
payments.to_sql("payments", conn, index=False, if_exists="replace")
reviews.to_sql("reviews", conn, index=False, if_exists="replace")

print("All tables loaded into SQL.")

All tables loaded into SQL.


In [ ]:
pd.read_sql_query("""
SELECT *
FROM orders
LIMIT 5;
""", conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
#Investigate the 775 unmatched orders
query = """
SELECT
    o.order_status,
    COUNT(*) AS orders_without_items
FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE oi.order_id IS NULL
GROUP BY o.order_status
ORDER BY orders_without_items DESC;
"""

pd.read_sql_query(query, conn)

,order_status,orders_without_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [ ]:
#Investigate the 8 orders that are not "unavailabel" or "canceled"
query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date
FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE oi.order_id IS NULL
  AND o.order_status NOT IN ('unavailable', 'canceled')
ORDER BY o.order_status;
"""

pd.read_sql_query(query, conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,b5359909123fa03c50bdb0cfed07f098,438449d4af8980d107bf04571413a8e7,created,2017-12-05 01:07:52,None,None,None
1,dba5062fbda3af4fb6c33b1e040ca38f,964a6df3d9bdf60fe3e7b8bb69ed893a,created,2018-02-09 17:21:04,None,None,None
2,7a4df5d8cff4090e541401a20a22bb80,725e9c75605414b21fd8c8d5a1c2f1d6,created,2017-11-25 11:10:33,None,None,None
3,35de4050331c6c644cddc86f4f2d0d64,4ee64f4bfc542546f422da0aeb462853,created,2017-12-05 01:07:58,None,None,None
4,90ab3e7d52544ec7bc3363c82689965f,7d61b9f4f216052ba664f22e9c504ef1,created,2017-11-06 13:12:34,None,None,None
5,2ce9683175cdab7d1c95bcbb3e36f478,b2d7ae0415dbbca535b5f7b38056dd1f,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39,None,None
6,e04f1da1f48bf2bbffcf57b9824f76e1,0d00d77134cae4c58695086ad8d85100,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38,None,None
7,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,None


In [ ]:
#Check one missing payment
query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_customer_date
FROM orders o
LEFT JOIN payments p
    ON o.order_id = p.order_id
WHERE p.order_id IS NULL;
"""

pd.read_sql_query(query, conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date
0,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-09 07:47:38


In [ ]:
#Investigate the 768 missing reviews
query = """
SELECT
    o.order_status,
    COUNT(*) AS orders_without_reviews
FROM orders o
LEFT JOIN reviews r
    ON o.order_id = r.order_id
WHERE r.order_id IS NULL
GROUP BY o.order_status
ORDER BY orders_without_reviews DESC;
"""

pd.read_sql_query(query, conn)

,order_status,orders_without_reviews
0,delivered,646
1,shipped,75
2,canceled,20
3,unavailable,14
4,processing,6
5,invoiced,5
6,created,2


How well is the delivery operation performing?

*   On-time vs. late deliveries
*   Average delivery time
*   Where delays are concentrated

Where delays are concentrated

* What factors are associated with poor customer satisfaction?
* Review scores
* Delivery delays
* Product categories
* Sellers
* Order characteristics

Where are the biggest operational problems?
* Sellers with high late-delivery rates
* Product categories with high issue rates
* Geographic patterns
* Canceled/unavailable orders

What should the business prioritize?
* Identify the areas with the greatest combination of volume + operational problems + customer dissatisfaction
* Translate those findings into specific recommendations

In [ ]:
#What percentage of delivered orders arrived late compared with the estimated delivery date?
query = """
SELECT
    COUNT(*) AS delivered_orders,
    SUM(
        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
"""

pd.read_sql_query(query, conn)

,delivered_orders,late_orders,late_delivery_rate
0,96470,7826,8.11


In [ ]:
#Analyze late deliveries by product category
query = """
SELECT
    COALESCE(p.product_category_name, 'Unknown') AS product_category,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY p.product_category_name
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,product_category,delivered_orders,late_orders,late_delivery_rate
0,artigos_de_natal,125,18,14.40
1,fashion_underwear_e_moda_praia,117,16,13.68
2,audio,348,46,13.22
3,construcao_ferramentas_iluminacao,242,30,12.40
4,moveis_escritorio,1254,149,11.88
5,livros_tecnicos,256,29,11.33
6,casa_conforto,392,44,11.22
7,alimentos,441,49,11.11
8,moveis_decoracao,6307,688,10.91
9,eletronicos,2517,266,10.57


In [ ]:
#Finding the biggest sources of late orders
query = """
SELECT
    COALESCE(p.product_category_name, 'Unknown') AS product_category,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY p.product_category_name
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY late_orders DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,product_category,delivered_orders,late_orders,late_delivery_rate
0,cama_mesa_banho,9272,920,9.92
1,beleza_saude,8647,857,9.91
2,moveis_decoracao,6307,688,10.91
3,esporte_lazer,7529,625,8.30
4,informatica_acessorios,6529,594,9.10
5,relogios_presentes,5493,485,8.83
6,utilidades_domesticas,5743,441,7.68
7,telefonia,4093,369,9.02
8,automotivo,3809,343,9.00
9,ferramentas_jardim,3448,340,9.86


In [ ]:
#Compare review scores for late vs. on-time orders
query = """
SELECT
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'Late'
        ELSE 'On Time'
    END AS delivery_performance,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    SUM(CASE WHEN r.review_score = 1 THEN 1 ELSE 0 END) AS one_star_reviews,
    ROUND(
        100.0 * SUM(CASE WHEN r.review_score = 1 THEN 1 ELSE 0 END)
        / COUNT(r.review_score),
        2
    ) AS one_star_rate
FROM orders o
JOIN reviews r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'Late'
        ELSE 'On Time'
    END;
"""

pd.read_sql_query(query, conn)

,delivery_performance,orders,avg_review_score,one_star_reviews,one_star_rate
0,Late,7661,2.57,3554,46.16
1,On Time,88163,4.29,5851,6.60


In [ ]:
#Find where late orders are most damaging
query = """
SELECT
    p.product_category_name AS product_category,
    COUNT(DISTINCT o.order_id) AS late_orders,
    ROUND(AVG(r.review_score), 2) AS avg_late_review_score,
    SUM(CASE WHEN r.review_score = 1 THEN 1 ELSE 0 END) AS one_star_reviews,
    ROUND(
        100.0 * SUM(CASE WHEN r.review_score = 1 THEN 1 ELSE 0 END)
        / COUNT(r.review_score),
        2
    ) AS one_star_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
JOIN reviews r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND o.order_delivered_customer_date > o.order_estimated_delivery_date
GROUP BY p.product_category_name
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY one_star_rate DESC;
"""

pd.read_sql_query(query, conn)

,product_category,late_orders,avg_late_review_score,one_star_reviews,one_star_rate
0,bebes,251,2.33,135,52.94
1,brinquedos,275,2.39,151,51.89
2,None,124,2.38,72,51.06
3,esporte_lazer,575,2.48,313,50.81
4,fashion_bolsas_e_acessorios,120,2.43,62,49.60
5,relogios_presentes,456,2.40,234,49.37
6,moveis_escritorio,112,2.48,72,48.98
7,cama_mesa_banho,788,2.50,437,48.23
8,ferramentas_jardim,270,2.57,162,48.21
9,informatica_acessorios,493,2.56,278,47.52


In [ ]:
#Let's investigate the operational cause
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY oi.seller_id
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,seller_id,delivered_orders,late_orders,late_delivery_rate
0,06a2c3af7b3aee5d69171b0e14f0ee87,389,95,24.42
1,88460e8ebdecbfecb5f9601833981930,246,59,23.98
2,2c9e548be18521d1c43cde1c582c6de8,124,29,23.39
3,1ca7077d890b907f89be8c954a02686a,108,25,23.15
4,cd68562d3f44870c08922d380acae552,122,26,21.31
...,...,...,...,...
205,6c7d50c24b3ccd2fd83b44d8bb34e073,112,2,1.79
206,0ea22c1cfbdc755f86b9b54b39c16043,234,4,1.71
207,12b9676b00f60f3b700e83af21824c0e,133,2,1.50
208,0bae85eb84b9fb3bd773911e89288d54,136,1,0.74


In [ ]:
#Identify where the high-risk sellers are located
query = """
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN sellers s
    ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY late_delivery_rate DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,seller_id,seller_city,seller_state,delivered_orders,late_orders,late_delivery_rate
0,06a2c3af7b3aee5d69171b0e14f0ee87,sao luis,MA,389,95,24.42
1,88460e8ebdecbfecb5f9601833981930,maringa,PR,246,59,23.98
2,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,SP,124,29,23.39
3,1ca7077d890b907f89be8c954a02686a,santana de parnaiba,SP,108,25,23.15
4,cd68562d3f44870c08922d380acae552,ribeirao preto,SP,122,26,21.31
5,e5a3438891c0bfdb9394643f95273d8e,limeira,SP,216,45,20.83
6,8160255418d5aaa7dbdc9f4c64ebda44,ibitinga,SP,380,76,20.00
7,b2479f944e1b90cf8a5de1bbfde284d6,ibitinga,SP,101,19,18.81
8,f7ba60f8c3f99e7ee4042fdef03b70c4,sao bernardo do campo,SP,218,38,17.43
9,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP,121,21,17.36


In [ ]:
#Test whether the problem is actually geographic
query = """
SELECT
    s.seller_state,
    COUNT(DISTINCT s.seller_id) AS sellers,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN sellers s
    ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY s.seller_state
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,seller_state,sellers,delivered_orders,late_orders,late_delivery_rate
0,MA,1,389,95,24.42
1,SP,1769,68635,6697,9.76
2,RJ,163,4227,380,8.99
3,ES,22,310,24,7.74
4,DF,30,808,59,7.30
5,PR,335,7512,548,7.29
6,SC,184,3603,235,6.52
7,BA,18,550,34,6.18
8,MG,236,7733,477,6.17
9,MT,4,136,7,5.15


In [ ]:
#Identify the highest-impact sellers
query = """
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN sellers s
    ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state
HAVING COUNT(DISTINCT o.order_id) >= 200
ORDER BY late_orders DESC
LIMIT 15;
"""

pd.read_sql_query(query, conn)

,seller_id,seller_city,seller_state,delivered_orders,late_orders,late_delivery_rate
0,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1772,214,12.08
1,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1399,182,13.01
2,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1124,133,11.83
3,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,910,131,14.40
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,973,130,13.36
5,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,1819,124,6.82
6,ea8482cd71df3c1969d7b9473ff13abc,sao paulo,SP,1132,123,10.87
7,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1261,119,9.44
8,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1311,113,8.62
9,8b321bb669392f5163d04c59e235e066,sao paulo,SP,930,103,11.08


In [ ]:
#Measuring the severity of delivery delays
query = """
WITH delivery_delays AS (
    SELECT
        order_id,
        julianday(order_delivered_customer_date)
        - julianday(order_estimated_delivery_date) AS days_late
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
      AND order_delivered_customer_date > order_estimated_delivery_date
)

SELECT
    CASE
        WHEN days_late < 4 THEN '1-3 days late'
        WHEN days_late < 8 THEN '4-7 days late'
        WHEN days_late < 15 THEN '8-14 days late'
        ELSE '15+ days late'
    END AS delay_bucket,
    COUNT(*) AS orders,
    ROUND(AVG(days_late), 1) AS avg_days_late
FROM delivery_delays
GROUP BY delay_bucket
ORDER BY
    CASE delay_bucket
        WHEN '1-3 days late' THEN 1
        WHEN '4-7 days late' THEN 2
        WHEN '8-14 days late' THEN 3
        WHEN '15+ days late' THEN 4
    END;
"""

pd.read_sql_query(query, conn)

,delay_bucket,orders,avg_days_late
0,1-3 days late,3162,1.8
1,4-7 days late,1802,6.2
2,8-14 days late,1478,11.3
3,15+ days late,1384,29.9


In [ ]:
#Analyze the fulfillment stages
query = """
SELECT
    CASE
        WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 'Late'
        ELSE 'On Time'
    END AS delivery_performance,

    ROUND(
        AVG(
            julianday(order_delivered_carrier_date)
            - julianday(order_approved_at)
        ),
        1
    ) AS avg_approval_to_carrier_days,

    ROUND(
        AVG(
            julianday(order_delivered_customer_date)
            - julianday(order_delivered_carrier_date)
        ),
        1
    ) AS avg_carrier_to_customer_days,

    ROUND(
        AVG(
            julianday(order_delivered_customer_date)
            - julianday(order_approved_at)
        ),
        1
    ) AS avg_total_fulfillment_days

FROM orders

WHERE order_status = 'delivered'
  AND order_approved_at IS NOT NULL
  AND order_delivered_carrier_date IS NOT NULL
  AND order_delivered_customer_date IS NOT NULL

GROUP BY
    CASE
        WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 'Late'
        ELSE 'On Time'
    END;
"""

pd.read_sql_query(query, conn)

,delivery_performance,avg_approval_to_carrier_days,avg_carrier_to_customer_days,avg_total_fulfillment_days
0,Late,5.3,25.7,31.0
1,On Time,2.6,7.9,10.5


In [ ]:
#Distribution of carrier transit time
query = """
WITH fulfillment AS (
    SELECT
        order_id,
        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 'Late'
            ELSE 'On Time'
        END AS delivery_performance,
        julianday(order_delivered_customer_date)
        - julianday(order_delivered_carrier_date) AS carrier_days
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_carrier_date IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
)

SELECT
    delivery_performance,
    CASE
        WHEN carrier_days < 5 THEN '<5 days'
        WHEN carrier_days < 10 THEN '5-9 days'
        WHEN carrier_days < 15 THEN '10-14 days'
        WHEN carrier_days < 20 THEN '15-19 days'
        ELSE '20+ days'
    END AS carrier_transit_bucket,
    COUNT(*) AS orders
FROM fulfillment
GROUP BY
    delivery_performance,
    carrier_transit_bucket
ORDER BY
    delivery_performance,
    CASE carrier_transit_bucket
        WHEN '<5 days' THEN 1
        WHEN '5-9 days' THEN 2
        WHEN '10-14 days' THEN 3
        WHEN '15-19 days' THEN 4
        WHEN '20+ days' THEN 5
    END;
"""

pd.read_sql_query(query, conn)

,delivery_performance,carrier_transit_bucket,orders
0,Late,<5 days,650
1,Late,5-9 days,619
2,Late,10-14 days,620
3,Late,15-19 days,912
4,Late,20+ days,5024
5,On Time,<5 days,28817
6,On Time,5-9 days,34841
7,On Time,10-14 days,15867
8,On Time,15-19 days,5743
9,On Time,20+ days,3376


In [ ]:
#Find where long carrier transit is concentrated
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS delivered_orders,

    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(
            julianday(o.order_delivered_customer_date)
            - julianday(o.order_delivered_carrier_date)
        ),
        1
    ) AS avg_carrier_to_customer_days

FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_carrier_date IS NOT NULL
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL

GROUP BY c.customer_state

HAVING COUNT(DISTINCT o.order_id) >= 100

ORDER BY avg_carrier_to_customer_days DESC;
"""

pd.read_sql_query(query, conn)

,customer_state,delivered_orders,late_orders,late_delivery_rate,avg_carrier_to_customer_days
0,AM,145,6,4.14,23.5
1,AL,397,95,23.93,21.1
2,PA,946,117,12.37,20.3
3,MA,717,141,19.67,18.0
4,SE,335,51,15.22,17.9
5,CE,1279,196,15.32,17.9
6,PB,517,57,11.03,16.9
7,RO,243,7,2.88,16.5
8,PI,476,76,15.97,16.3
9,BA,3256,457,14.04,16.0


In [ ]:
#Analyze seller state to customer state routes
query = """
SELECT
    s.seller_state,
    c.customer_state,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate,

    ROUND(
        AVG(
            julianday(o.order_delivered_customer_date)
            - julianday(o.order_delivered_carrier_date)
        ),
        1
    ) AS avg_carrier_to_customer_days

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN sellers s
    ON oi.seller_id = s.seller_id

JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_carrier_date IS NOT NULL
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL

GROUP BY
    s.seller_state,
    c.customer_state

HAVING COUNT(DISTINCT o.order_id) >= 100

ORDER BY late_delivery_rate DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,seller_state,customer_state,delivered_orders,late_orders,late_delivery_rate,avg_carrier_to_customer_days
0,MA,SP,124,35,28.23,10.6
1,SP,AL,256,69,26.95,21.2
2,SP,MA,493,122,24.75,18.4
3,SP,SE,208,43,20.67,17.7
4,SP,PI,329,65,19.76,16.9
5,PR,BA,144,25,17.36,17.5
6,SP,CE,970,168,17.32,17.6
7,SP,RJ,8188,1396,17.05,12.6
8,SP,BA,2313,388,16.77,16.3
9,SP,MS,478,75,15.69,12.7


In [ ]:
#Analyze delivery performance over time
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS purchase_month,

    COUNT(DISTINCT order_id) AS delivered_orders,

    SUM(
        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT order_id),
        2
    ) AS late_delivery_rate

FROM orders

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL

GROUP BY purchase_month

ORDER BY purchase_month;
"""

pd.read_sql_query(query, conn)

,purchase_month,delivered_orders,late_orders,late_delivery_rate
0,2016-09,1,1,100.00
1,2016-10,265,3,1.13
2,2016-12,1,0,0.00
3,2017-01,750,23,3.07
4,2017-02,1653,53,3.21
5,2017-03,2546,142,5.58
6,2017-04,2303,181,7.86
7,2017-05,3545,128,3.61
8,2017-06,3135,121,3.86
9,2017-07,3872,133,3.43


In [ ]:
#Comparing high-delay vs normal months
query = """
SELECT
    s.seller_state,
    c.customer_state,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN sellers s
    ON oi.seller_id = s.seller_id

JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-03'

GROUP BY
    s.seller_state,
    c.customer_state

HAVING COUNT(DISTINCT o.order_id) >= 50

ORDER BY late_orders DESC

LIMIT 20;
"""

pd.read_sql_query(query, conn)

,seller_state,customer_state,delivered_orders,late_orders,late_delivery_rate
0,SP,RJ,591,284,48.05
1,SP,SP,2326,282,12.12
2,SP,MG,590,175,29.66
3,SP,BA,177,79,44.63
4,SP,RS,272,65,23.90
5,SP,ES,103,58,56.31
6,SP,CE,76,53,69.74
7,SP,SC,171,46,26.90
8,SP,PR,247,45,18.22
9,RJ,SP,86,43,50.00


In [ ]:
#Compare Reviews the high risk routes in March
query = """
SELECT
    s.seller_state,
    c.customer_state,

    COUNT(DISTINCT o.order_id) AS orders,

    SUM(
        CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT o.order_id),
        2
    ) AS late_rate,

    ROUND(AVG(r.review_score), 2) AS avg_review_score,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN r.review_score = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(r.review_score),
        2
    ) AS one_star_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN sellers s
    ON oi.seller_id = s.seller_id

JOIN customers c
    ON o.customer_id = c.customer_id

JOIN reviews r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-03'

GROUP BY
    s.seller_state,
    c.customer_state

HAVING COUNT(DISTINCT o.order_id) >= 50

ORDER BY late_rate DESC
LIMIT 15;
"""

pd.read_sql_query(query, conn)

,seller_state,customer_state,orders,late_orders,late_rate,avg_review_score,one_star_rate
0,SP,CE,75,52,69.33,3.11,31.52
1,SP,PA,52,31,59.62,2.47,51.72
2,SP,ES,100,57,57.00,3.08,33.33
3,RJ,SP,83,40,48.19,2.96,43.33
4,SP,RJ,580,273,47.07,2.88,40.72
5,SP,BA,177,79,44.63,3.28,28.10
6,PR,RJ,82,28,34.15,3.63,21.10
7,SP,MG,584,172,29.45,3.51,23.56
8,SP,DF,105,29,27.62,3.59,24.79
9,SP,SC,169,46,27.22,3.46,19.79


In [ ]:
#Building an order-level delivery operations table
query = """
CREATE TABLE delivery_operations AS

SELECT
    o.order_id,
    o.customer_id,
    c.customer_state,

    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    -- Delivery performance
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'Late'
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 'On Time'
        ELSE 'Unknown'
    END AS delivery_performance,

    -- Days late
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_estimated_delivery_date),
                1
            )
        ELSE 0
    END AS days_late,

    -- Approval to carrier
    CASE
        WHEN o.order_approved_at IS NOT NULL
         AND o.order_delivered_carrier_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_carrier_date)
                - julianday(o.order_approved_at),
                1
            )
        ELSE NULL
    END AS approval_to_carrier_days,

    -- Carrier to customer
    CASE
        WHEN o.order_delivered_carrier_date IS NOT NULL
         AND o.order_delivered_customer_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_delivered_carrier_date),
                1
            )
        ELSE NULL
    END AS carrier_to_customer_days,

    -- Total fulfillment
    CASE
        WHEN o.order_approved_at IS NOT NULL
         AND o.order_delivered_customer_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_approved_at),
                1
            )
        ELSE NULL
    END AS total_fulfillment_days,

    -- Delay bucket
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 3
            THEN '1-3 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 7
            THEN '4-7 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 14
            THEN '8-14 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN '15+ days late'

        ELSE 'On Time'
    END AS delay_bucket,

    -- Customer review
    r.review_score,

    CASE
        WHEN r.review_score = 1 THEN 1
        ELSE 0
    END AS one_star_flag

FROM orders o

LEFT JOIN customers c
    ON o.customer_id = c.customer_id

LEFT JOIN reviews r
    ON o.order_id = r.order_id;
"""

conn.execute(query)
conn.commit()

print("delivery_operations table created successfully.")

delivery_operations table created successfully.


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM delivery_operations;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_orders
0,99992,99441


In [ ]:
query = """
SELECT
    order_id,
    COUNT(*) AS row_count
FROM delivery_operations
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC;
"""

pd.read_sql_query(query, conn)

,order_id,row_count
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2
...,...,...
542,029863af4b968de1e5d6a82782e662f5,2
543,02355020fd0a40a0d56df9f6ff060413,2
544,0176a6846bcb3b0d3aa3116a9a768597,2
545,013056cfe49763c6f66bda03396c5ee3,2


In [ ]:
query = """
SELECT
    COUNT(*) AS orders_with_duplicates,
    SUM(row_count - 1) AS extra_rows
FROM (
    SELECT
        order_id,
        COUNT(*) AS row_count
    FROM delivery_operations
    GROUP BY order_id
    HAVING COUNT(*) > 1
);
"""

pd.read_sql_query(query, conn)

,orders_with_duplicates,extra_rows
0,547,551


In [ ]:
conn.execute("DROP TABLE delivery_operations;")
conn.commit()

print("Old delivery_operations table removed.")

Old delivery_operations table removed.


In [ ]:
query = """
CREATE TABLE delivery_operations AS

WITH review_summary AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score,
        MAX(
            CASE
                WHEN review_score = 1 THEN 1
                ELSE 0
            END
        ) AS one_star_flag
    FROM reviews
    GROUP BY order_id
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_state,

    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'Late'
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 'On Time'
        ELSE 'Unknown'
    END AS delivery_performance,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_estimated_delivery_date),
                1
            )
        ELSE 0
    END AS days_late,

    CASE
        WHEN o.order_approved_at IS NOT NULL
         AND o.order_delivered_carrier_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_carrier_date)
                - julianday(o.order_approved_at),
                1
            )
        ELSE NULL
    END AS approval_to_carrier_days,

    CASE
        WHEN o.order_delivered_carrier_date IS NOT NULL
         AND o.order_delivered_customer_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_delivered_carrier_date),
                1
            )
        ELSE NULL
    END AS carrier_to_customer_days,

    CASE
        WHEN o.order_approved_at IS NOT NULL
         AND o.order_delivered_customer_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_approved_at),
                1
            )
        ELSE NULL
    END AS total_fulfillment_days,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 3
            THEN '1-3 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 7
            THEN '4-7 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 14
            THEN '8-14 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN '15+ days late'

        ELSE 'On Time'
    END AS delay_bucket,

    rs.review_score,
    COALESCE(rs.one_star_flag, 0) AS one_star_flag

FROM orders o

LEFT JOIN customers c
    ON o.customer_id = c.customer_id

LEFT JOIN review_summary rs
    ON o.order_id = rs.order_id;
"""

conn.execute(query)
conn.commit()

print("delivery_operations rebuilt successfully.")

delivery_operations rebuilt successfully.


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM delivery_operations;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_orders
0,99441,99441


In [ ]:
query = """
CREATE TABLE order_item_summary AS

SELECT
    oi.order_id,

    COUNT(*) AS item_count,

    COUNT(DISTINCT oi.seller_id) AS seller_count,

    COUNT(DISTINCT p.product_category_name) AS product_category_count,

    ROUND(SUM(oi.price), 2) AS total_product_value,

    ROUND(SUM(oi.freight_value), 2) AS total_freight_value

FROM order_items oi

LEFT JOIN products p
    ON oi.product_id = p.product_id

GROUP BY oi.order_id;
"""

conn.execute(query)
conn.commit()

print("order_item_summary created successfully.")

order_item_summary created successfully.


In [ ]:
query = """
SELECT
    COUNT(*) AS summary_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM order_item_summary;
"""

pd.read_sql_query(query, conn)

,summary_rows,unique_orders
0,98666,98666


In [ ]:
query = """
ALTER TABLE delivery_operations
ADD COLUMN item_count INTEGER;

ALTER TABLE delivery_operations
ADD COLUMN seller_count INTEGER;

ALTER TABLE delivery_operations
ADD COLUMN product_category_count INTEGER;

ALTER TABLE delivery_operations
ADD COLUMN total_product_value REAL;

ALTER TABLE delivery_operations
ADD COLUMN total_freight_value REAL;
"""

for statement in query.strip().split(";"):
    if statement.strip():
        conn.execute(statement)

conn.commit()

print("Item summary columns added.")

Item summary columns added.


In [ ]:
conn.execute("DROP TABLE delivery_operations;")
conn.commit()

print("Old delivery_operations table removed.")

Old delivery_operations table removed.


In [ ]:
query = """
CREATE TABLE delivery_operations AS

WITH review_summary AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score,
        MAX(
            CASE
                WHEN review_score = 1 THEN 1
                ELSE 0
            END
        ) AS one_star_flag
    FROM reviews
    GROUP BY order_id
),

order_item_summary AS (
    SELECT
        oi.order_id,
        COUNT(*) AS item_count,
        COUNT(DISTINCT oi.seller_id) AS seller_count,
        COUNT(DISTINCT p.product_category_name) AS product_category_count,
        ROUND(SUM(oi.price), 2) AS total_product_value,
        ROUND(SUM(oi.freight_value), 2) AS total_freight_value
    FROM order_items oi
    LEFT JOIN products p
        ON oi.product_id = p.product_id
    GROUP BY oi.order_id
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_state,

    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'Late'
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date
            THEN 'On Time'
        ELSE 'Unknown'
    END AS delivery_performance,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_estimated_delivery_date),
                1
            )
        ELSE 0
    END AS days_late,

    CASE
        WHEN o.order_approved_at IS NOT NULL
         AND o.order_delivered_carrier_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_carrier_date)
                - julianday(o.order_approved_at),
                1
            )
        ELSE NULL
    END AS approval_to_carrier_days,

    CASE
        WHEN o.order_delivered_carrier_date IS NOT NULL
         AND o.order_delivered_customer_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_delivered_carrier_date),
                1
            )
        ELSE NULL
    END AS carrier_to_customer_days,

    CASE
        WHEN o.order_approved_at IS NOT NULL
         AND o.order_delivered_customer_date IS NOT NULL
            THEN ROUND(
                julianday(o.order_delivered_customer_date)
                - julianday(o.order_approved_at),
                1
            )
        ELSE NULL
    END AS total_fulfillment_days,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 3
            THEN '1-3 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 7
            THEN '4-7 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             AND julianday(o.order_delivered_customer_date)
                 - julianday(o.order_estimated_delivery_date) <= 14
            THEN '8-14 days late'

        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN '15+ days late'

        ELSE 'On Time'
    END AS delay_bucket,

    rs.review_score,
    COALESCE(rs.one_star_flag, 0) AS one_star_flag,

    s.item_count,
    s.seller_count,
    s.product_category_count,
    s.total_product_value,
    s.total_freight_value

FROM orders o

LEFT JOIN customers c
    ON o.customer_id = c.customer_id

LEFT JOIN review_summary rs
    ON o.order_id = rs.order_id

LEFT JOIN order_item_summary s
    ON o.order_id = s.order_id;
"""

conn.execute(query)
conn.commit()

print("delivery_operations rebuilt successfully.")

delivery_operations rebuilt successfully.


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(item_count) AS orders_with_items,
    COUNT(*) - COUNT(item_count) AS orders_without_items
FROM delivery_operations;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_orders,orders_with_items,orders_without_items
0,99441,99441,98666,775


In [ ]:
#Create order_routes
query = """
CREATE TABLE order_routes AS

SELECT DISTINCT
    oi.order_id,
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_state

FROM order_items oi

LEFT JOIN sellers s
    ON oi.seller_id = s.seller_id

LEFT JOIN orders o
    ON oi.order_id = o.order_id

LEFT JOIN customers c
    ON o.customer_id = c.customer_id;
"""

conn.execute(query)
conn.commit()

print("order_routes created successfully.")

order_routes created successfully.


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT seller_id) AS unique_sellers
FROM order_routes;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_orders,unique_sellers
0,100010,98666,3095


In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN seller_state IS NULL THEN 1 ELSE 0 END) AS missing_seller_state,
    SUM(CASE WHEN customer_state IS NULL THEN 1 ELSE 0 END) AS missing_customer_state
FROM order_routes;
"""

pd.read_sql_query(query, conn)

,total_rows,missing_seller_state,missing_customer_state
0,100010,0,0


In [ ]:
#quality check
query = """
SELECT
    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN delivery_performance = 'Late'
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    SUM(
        CASE
            WHEN delivery_performance = 'On Time'
            THEN 1
            ELSE 0
        END
    ) AS on_time_orders,

    SUM(
        CASE
            WHEN days_late < 0
            THEN 1
            ELSE 0
        END
    ) AS negative_days_late,

    SUM(
        CASE
            WHEN total_product_value < 0
            THEN 1
            ELSE 0
        END
    ) AS negative_product_value,

    SUM(
        CASE
            WHEN total_freight_value < 0
            THEN 1
            ELSE 0
        END
    ) AS negative_freight_value

FROM delivery_operations;
"""

pd.read_sql_query(query, conn)

,total_orders,late_orders,on_time_orders,negative_days_late,negative_product_value,negative_freight_value
0,99441,7827,88649,0,0,0


In [ ]:
# Export the tables
# Export the order-level table
pd.read_sql_query(
    "SELECT * FROM delivery_operations",
    conn
).to_csv("/content/delivery_operations.csv", index=False)

# Export the route-level table
pd.read_sql_query(
    "SELECT * FROM order_routes",
    conn
).to_csv("/content/order_routes.csv", index=False)

print("Both tables exported successfully.")

Both tables exported successfully.


In [ ]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'delivery_operations.csv', 'order_routes.csv', 'olist', 'archive (1).zip']


In [ ]:
from google.colab import files

files.download("/content/delivery_operations.csv")
files.download("/content/order_routes.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(orders["order_purchase_timestamp"].head())
print(orders["order_purchase_timestamp"].dtype)

0    2017-10-02 10:56:33
1    2018-07-24 20:41:37
2    2018-08-08 08:38:49
3    2017-11-18 19:28:06
4    2018-02-13 21:18:39
Name: order_purchase_timestamp, dtype: object
object


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

In [ ]:
print(orders["order_purchase_timestamp"].dtype)

datetime64[ns]


In [ ]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
print(orders[["order_purchase_timestamp", "purchase_month"]].head())

  order_purchase_timestamp purchase_month
0      2017-10-02 10:56:33        2017-10
1      2018-07-24 20:41:37        2018-07
2      2018-08-08 08:38:49        2018-08
3      2017-11-18 19:28:06        2017-11
4      2018-02-13 21:18:39        2018-02


In [ ]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_month']


In [ ]:
# Start with the orders table
delivery_operations = orders.copy()

# Add customer state
delivery_operations = delivery_operations.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

# Calculate delivery metrics
delivery_operations["delivery_performance"] = "Other"

delivered = delivery_operations["order_status"] == "delivered"

delivery_operations.loc[delivered, "days_late"] = (
    pd.to_datetime(delivery_operations.loc[delivered, "order_delivered_customer_date"])
    - pd.to_datetime(delivery_operations.loc[delivered, "order_estimated_delivery_date"])
).dt.total_seconds() / 86400

delivery_operations.loc[
    delivered & (delivery_operations["days_late"] > 0),
    "delivery_performance"
] = "Late"

delivery_operations.loc[
    delivered & (delivery_operations["days_late"] <= 0),
    "delivery_performance"
] = "On Time"

# Approval → carrier
delivery_operations["approval_to_carrier_days"] = (
    pd.to_datetime(delivery_operations["order_delivered_carrier_date"])
    - pd.to_datetime(delivery_operations["order_approved_at"])
).dt.total_seconds() / 86400

# Carrier → customer
delivery_operations["carrier_to_customer_days"] = (
    pd.to_datetime(delivery_operations["order_delivered_customer_date"])
    - pd.to_datetime(delivery_operations["order_delivered_carrier_date"])
).dt.total_seconds() / 86400

# Total fulfillment
delivery_operations["total_fulfillment_days"] = (
    pd.to_datetime(delivery_operations["order_delivered_customer_date"])
    - pd.to_datetime(delivery_operations["order_purchase_timestamp"])
).dt.total_seconds() / 86400

In [ ]:
# Order item information
item_summary = (
    order_items.groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        seller_count=("seller_id", "nunique"),
        total_product_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .reset_index()
)

# Product category count
order_items_products = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

category_summary = (
    order_items_products.groupby("order_id")
    .agg(
        product_category_count=("product_category_name", "nunique")
    )
    .reset_index()
)

# Add item metrics
delivery_operations = delivery_operations.merge(
    item_summary,
    on="order_id",
    how="left"
)

delivery_operations = delivery_operations.merge(
    category_summary,
    on="order_id",
    how="left"
)

In [ ]:
review_summary = (
    reviews.groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

delivery_operations = delivery_operations.merge(
    review_summary,
    on="order_id",
    how="left"
)

delivery_operations["one_star_flag"] = (
    delivery_operations["review_score"] == 1
).astype(int)

In [ ]:
delivery_operations["delay_bucket"] = pd.cut(
    delivery_operations["days_late"],
    bins=[0, 3, 7, 14, float("inf")],
    labels=["1-3 days late", "4-7 days late", "8-14 days late", "15+ days late"],
    include_lowest=False
)

delivery_operations["delay_bucket"] = (
    delivery_operations["delay_bucket"]
    .astype("object")
    .where(
        delivery_operations["days_late"] > 0,
        None
    )
)

In [ ]:
print(delivery_operations.shape)
print(delivery_operations.columns.tolist())

(99441, 23)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_month', 'customer_state', 'delivery_performance', 'days_late', 'approval_to_carrier_days', 'carrier_to_customer_days', 'total_fulfillment_days', 'item_count', 'seller_count', 'total_product_value', 'total_freight_value', 'product_category_count', 'review_score', 'one_star_flag', 'delay_bucket']


In [ ]:
print(
    delivery_operations["delivery_performance"].value_counts(dropna=False)
)

print(
    delivery_operations["purchase_month"].value_counts().sort_index()
)

delivery_performance
On Time    88644
Late        7826
Other       2971
Name: count, dtype: int64
purchase_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Name: count, dtype: int64


In [ ]:
delivery_operations.to_csv(
    "/content/delivery_operations_looker.csv",
    index=False
)

print("Export complete!")

Export complete!


In [ ]:
# Create seller-level delivery performance table

# Start with delivered orders that have a valid delivery date
delivered_orders = delivery_operations[
    delivery_operations["order_delivered_customer_date"].notna()
].copy()

# Connect each order to its seller
seller_orders = delivered_orders[
    ["order_id", "delivery_performance"]
].merge(
    order_items[["order_id", "seller_id"]],
    on="order_id",
    how="inner"
)

# Remove duplicate order/seller combinations
seller_orders = seller_orders.drop_duplicates(
    subset=["order_id", "seller_id"]
)

# Aggregate to seller level
seller_performance = (
    seller_orders
    .groupby("seller_id", as_index=False)
    .agg(
        delivered_orders=("order_id", "nunique"),
        late_orders=(
            "delivery_performance",
            lambda x: (x == "Late").sum()
        )
    )
)

# Calculate late delivery rate
seller_performance["late_delivery_rate"] = (
    seller_performance["late_orders"] /
    seller_performance["delivered_orders"] * 100
)

# Add seller location
seller_performance = seller_performance.merge(
    sellers[["seller_id", "seller_city", "seller_state"]],
    on="seller_id",
    how="left"
)

# Sort highest to lowest late delivery rate
seller_performance = seller_performance.sort_values(
    "late_delivery_rate",
    ascending=False
)

seller_performance.head(20)

,seller_id,delivered_orders,late_orders,late_delivery_rate,seller_city,seller_state
2850,f524ad65d7e0f1daab730ef2d2e86196,1,1,100.0,soledade,RS
951,51a04a8a6bdcb23deccc82b0b80742cf,1,1,100.0,braganca paulista,SP
2859,f5fea3ffed6c2e889bab72705557c63a,1,1,100.0,rio de janeiro,RJ
649,391bbd13b6452244774beff1824006ed,1,1,100.0,campinas,SP
288,19484c79cef6c062cb177aa4ef2fcc3c,1,1,100.0,valinhos,SP
1835,9c57bc60cfad5ee62d35d3f1ce4593a1,1,1,100.0,curitiba,PR
1827,9bf11dfc0bec77e5a23028043c3c5a8f,1,1,100.0,contagem,MG
382,20f0aeea30bc3b8c4420be8ced4226c0,1,1,100.0,santa barbara d'oeste,SP
2184,bc8c8d665ec4664d286be0d521722b19,1,1,100.0,sao paulo,SP
1865,9ef932e837d8b7f392c0bfee9d359dc2,1,1,100.0,ibitinga,SP


In [ ]:
print("Unique sellers:", seller_performance["seller_id"].nunique())
print("Rows:", len(seller_performance))

Unique sellers: 2970
Rows: 2970


In [ ]:
seller_performance[
    seller_performance["delivered_orders"] >= 100
].sort_values(
    "late_delivery_rate",
    ascending=False
).head(20)

,seller_id,delivered_orders,late_orders,late_delivery_rate,seller_city,seller_state
79,06a2c3af7b3aee5d69171b0e14f0ee87,389,90,23.136247,sao luis,MA
323,1ca7077d890b907f89be8c954a02686a,108,24,22.222222,santana de parnaiba,SP
1606,88460e8ebdecbfecb5f9601833981930,246,48,19.512195,maringa,PR
2657,e5a3438891c0bfdb9394643f95273d8e,216,40,18.518519,limeira,SP
2404,cd68562d3f44870c08922d380acae552,122,22,18.032787,ribeirao preto,SP
1532,8160255418d5aaa7dbdc9f4c64ebda44,380,63,16.578947,ibitinga,SP
513,2c9e548be18521d1c43cde1c582c6de8,124,20,16.129032,mogi das cruzes,SP
2577,dd7ddc04e1b6c2c614352b383efe2d36,121,19,15.702479,sao paulo,SP
2877,f7ba60f8c3f99e7ee4042fdef03b70c4,218,34,15.596330,sao bernardo do campo,SP
781,431af27f296bc6519d890aa5a05fdb11,116,18,15.517241,ribeirao preto,SP


In [ ]:
print("Missing seller IDs:", seller_performance["seller_id"].isna().sum())
print("Missing cities:", seller_performance["seller_city"].isna().sum())
print("Missing states:", seller_performance["seller_state"].isna().sum())
print("Rates over 100%:", (seller_performance["late_delivery_rate"] > 100).sum())

Missing seller IDs: 0
Missing cities: 0
Missing states: 0
Rates over 100%: 0


In [ ]:
# Create volume flag for Looker
seller_performance["volume_tier"] = seller_performance[
    "delivered_orders"
].apply(
    lambda x: "100+ Orders" if x >= 100 else "Under 100 Orders"
)

# Reorder columns
seller_performance = seller_performance[
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "delivered_orders",
        "late_orders",
        "late_delivery_rate",
        "volume_tier"
    ]
]

# Sort by late delivery rate
seller_performance = seller_performance.sort_values(
    "late_delivery_rate",
    ascending=False
)

# Final check
print("Rows:", len(seller_performance))
print("Columns:", seller_performance.columns.tolist())

seller_performance.head(10)

Rows: 2970
Columns: ['seller_id', 'seller_city', 'seller_state', 'delivered_orders', 'late_orders', 'late_delivery_rate', 'volume_tier']


,seller_id,seller_city,seller_state,delivered_orders,late_orders,late_delivery_rate,volume_tier
1525,80ceebb4ee9b31afb6c6a916a574a1e2,londrina,PR,1,1,100.0,Under 100 Orders
2850,f524ad65d7e0f1daab730ef2d2e86196,soledade,RS,1,1,100.0,Under 100 Orders
951,51a04a8a6bdcb23deccc82b0b80742cf,braganca paulista,SP,1,1,100.0,Under 100 Orders
912,4e42581f08e8cfc7c090f930bac4552a,portoferreira,SP,1,1,100.0,Under 100 Orders
1850,9da15f4a4ea758d9eeb49000dbe57e22,osasco,SP,1,1,100.0,Under 100 Orders
1865,9ef932e837d8b7f392c0bfee9d359dc2,ibitinga,SP,1,1,100.0,Under 100 Orders
2184,bc8c8d665ec4664d286be0d521722b19,sao paulo,SP,1,1,100.0,Under 100 Orders
382,20f0aeea30bc3b8c4420be8ced4226c0,santa barbara d'oeste,SP,1,1,100.0,Under 100 Orders
1827,9bf11dfc0bec77e5a23028043c3c5a8f,contagem,MG,1,1,100.0,Under 100 Orders
1835,9c57bc60cfad5ee62d35d3f1ce4593a1,curitiba,PR,1,1,100.0,Under 100 Orders


In [ ]:
seller_performance.to_csv(
    "seller_performance.csv",
    index=False
)

print("Export complete: seller_performance.csv")

Export complete: seller_performance.csv


In [ ]:
# Create seller → customer route performance table

route_data = delivery_operations[
    [
        "order_id",
        "customer_state",
        "delivery_performance",
        "total_fulfillment_days"
    ]
].merge(
    order_items[["order_id", "seller_id"]],
    on="order_id",
    how="inner"
).merge(
    sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)

# One row per order + seller
route_data = route_data.drop_duplicates(
    subset=["order_id", "seller_id"]
)

# Keep delivered orders only
route_data = route_data[
    route_data["delivery_performance"].isin(["Late", "On Time"])
]

# Aggregate seller → customer state routes
route_performance = (
    route_data
    .groupby(
        ["seller_state", "customer_state"],
        as_index=False
    )
    .agg(
        delivered_orders=("order_id", "nunique"),
        late_orders=(
            "delivery_performance",
            lambda x: (x == "Late").sum()
        ),
        avg_fulfillment_days=("total_fulfillment_days", "mean")
    )
)

route_performance["late_delivery_rate"] = (
    route_performance["late_orders"] /
    route_performance["delivered_orders"] * 100
)

# Only keep routes with meaningful volume
route_performance = route_performance[
    route_performance["delivered_orders"] >= 50
]

route_performance = route_performance.sort_values(
    "late_delivery_rate",
    ascending=False
)

print(route_performance.head(20))
print("\nRows:", len(route_performance))

    seller_state customer_state  delivered_orders  late_orders  \
386           SP             AL               256           67   
142           MA             SP               124           31   
285           RJ             CE                54           12   
394           SP             MA               493          105   
401           SP             PI               329           60   
257           PR             BA               144           24   
409           SP             SE               208           34   
153           MG             MA                64           10   
403           SP             RJ              8188         1270   
390           SP             CE               970          148   
389           SP             BA              2313          345   
396           SP             MS               478           70   
392           SP             ES              1467          208   
258           PR             CE                64            9   
271       

In [ ]:
route_performance.to_csv(
    "route_performance.csv",
    index=False
)

In [ ]:
top_sellers = seller_performance[
    seller_performance["volume_tier"] == "100+ Orders"
].copy()

top_sellers = (
    top_sellers
    .sort_values("late_delivery_rate", ascending=False)
    .head(10)
)

top_sellers

,seller_id,seller_city,seller_state,delivered_orders,late_orders,late_delivery_rate,volume_tier
79,06a2c3af7b3aee5d69171b0e14f0ee87,sao luis,MA,389,90,23.136247,100+ Orders
323,1ca7077d890b907f89be8c954a02686a,santana de parnaiba,SP,108,24,22.222222,100+ Orders
1606,88460e8ebdecbfecb5f9601833981930,maringa,PR,246,48,19.512195,100+ Orders
2657,e5a3438891c0bfdb9394643f95273d8e,limeira,SP,216,40,18.518519,100+ Orders
2404,cd68562d3f44870c08922d380acae552,ribeirao preto,SP,122,22,18.032787,100+ Orders
1532,8160255418d5aaa7dbdc9f4c64ebda44,ibitinga,SP,380,63,16.578947,100+ Orders
513,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,SP,124,20,16.129032,100+ Orders
2577,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP,121,19,15.702479,100+ Orders
2877,f7ba60f8c3f99e7ee4042fdef03b70c4,sao bernardo do campo,SP,218,34,15.596330,100+ Orders
781,431af27f296bc6519d890aa5a05fdb11,ribeirao preto,SP,116,18,15.517241,100+ Orders


In [ ]:
top_sellers.to_csv(
    "top_10_sellers.csv",
    index=False
)

In [ ]:
# Brazil state lookup table for Looker Studio Geo Map

brazil_states = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins"
}

brazil_states_df = pd.DataFrame(
    list(brazil_states.items()),
    columns=["customer_state", "state_name"]
)

# Check the table
brazil_states_df

,customer_state,state_name
0,AC,Acre
1,AL,Alagoas
2,AP,Amapá
3,AM,Amazonas
4,BA,Bahia
5,CE,Ceará
6,DF,Distrito Federal
7,ES,Espírito Santo
8,GO,Goiás
9,MA,Maranhão


In [ ]:
brazil_states_df.to_csv("brazil_states_lookup.csv", index=False)

print("Created brazil_states_lookup.csv")

Created brazil_states_lookup.csv
